# 🧠 AFFERENT URL Threat Detection — Production Notebook v5

> **Dataset:** [URL-AF on Kaggle](https://www.kaggle.com/datasets/w5lasmbc/url-af)
> **Classes:** 4-way `benign` / `phishing` / `malware` / `other`
> **Goal:** one production-grade model, chosen empirically (classical ML vs deep learning), with a
> mandatory classical-model comparison kept for reference/explainability.

## Apa yang beda dari v4

v4 (cascade 3-tier + stacker) sudah dijalankan nyata di Kaggle. Hasilnya membongkar dua masalah:
cascade (baik versi lama maupun versi kalibrasi-ku) **lebih buruk** dari menjalankan semua tier
sekaligus, dan itu karena (a) gating biner P(malicious) membuang info "kelas spesifik mana",
dan (b) kalibrasi + stacker + pemilihan threshold semuanya di-fit di validation set yang sama →
sirkular, terlihat bagus di val (gap 0.87pp) tapi jebol di test (gap 12.3pp).

v5 membuang seluruh kerumitan cascade/stacker itu. Sebagai gantinya:

1. **Satu model produksi**, dipilih **berdasarkan angka**, bukan diasumsikan — kalau deep learning
   menang (sangat mungkin, dari 2 run sebelumnya DL konsisten >80% vs classical ML <65%), ya deep
   learning yang dipakai, penuh, tanpa tier tambahan.
2. **Bagian "Comparable Models" (baseline klasik) tetap ada** dan wajib — bukan cuma formalitas,
   tapi jadi bukti kuantitatif *kenapa* DL yang dipilih, dan XGBoost-nya tetap berguna untuk
   *feature importance* / penjelasan ke analis meski bukan model produksi.
3. **AfferentNet** — satu jaringan DL yang menyatukan 3 sinyal (karakter URL, fitur lexical
   tangan-buatan, TF-IDF char n-gram) dalam satu model yang dilatih end-to-end. Ini idealnya
   dapat manfaat kombinasi tiga sinyal itu (seperti stacker v4) TANPA overhead kalibrasi
   terpisah dan tanpa sirkularitas val yang menjebak v4.
4. **Kalibrasi & threshold di-split** (val dibagi val-untuk-kalibrasi vs val-untuk-threshold) —
   perbaikan langsung dari bug metodologi yang ditemukan di v4.
5. **Uji ketahanan adversarial** (perturbasi homoglyph, subdomain, query noise) — gap yang
   sebelumnya belum pernah diuji sama sekali.
6. Investigasi kelas "other" via crosstab source×label + clustering — supaya kita tahu isinya
   apa, bukan sekadar menebak "judol" seperti asumsi awal yang ternyata salah.

---


---
## §0 — Setup, Data Loading & Validation

In [ ]:
!pip install -q tldextract optuna xgboost scikit-learn statsmodels

In [ ]:
import os, re, math, hashlib, warnings, time, json, pickle
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from urllib.parse import urlparse

import tldextract, optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.isotonic import IsotonicRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, average_precision_score, roc_curve,
    f1_score, precision_score, recall_score, accuracy_score
)
from sklearn.preprocessing import LabelEncoder, label_binarize, MinMaxScaler, StandardScaler
from statsmodels.stats.contingency_tables import mcnemar as mcnemar_test

import xgboost as xgb
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")
sns.set_theme(style="darkgrid", palette="muted")
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE} | PyTorch: {torch.__version__} | XGBoost: {xgb.__version__}")

In [ ]:
# Dataset: https://www.kaggle.com/datasets/w5lasmbc/url-af
candidates = [
    Path("/kaggle/input/url-af"),
    Path("/kaggle/input/url-af/processed"),
    Path("/kaggle/input/datasets/w5lasmbc/url-af"),
    Path("../dataset/processed"),
    Path("dataset/processed"),
    Path("./"),
]
DATA_DIR = None
for c in candidates:
    if (c / "train.csv").exists():
        DATA_DIR = c
        break
if DATA_DIR is None and Path("/kaggle/input").exists():
    found = list(Path("/kaggle/input").rglob("train.csv"))
    if found:
        DATA_DIR = found[0].parent
if DATA_DIR is None:
    DATA_DIR = Path("/kaggle/input/url-af")

print(f"Detected DATA_DIR: {DATA_DIR}")

if Path("/kaggle/working").exists():
    OUT_DIR = Path("/kaggle/working/afferent_models")
else:
    OUT_DIR = Path("./afferent_models")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Artifacts will be saved to: {OUT_DIR}")

train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df   = pd.read_csv(DATA_DIR / "val.csv")
test_df  = pd.read_csv(DATA_DIR / "test.csv")
ood_df   = pd.read_csv(DATA_DIR / "indonesian_threat_casestudy.csv")

print(f"\nTrain : {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}  OOD: {len(ood_df):,}")
print("Columns:", train_df.columns.tolist())
print("\nLabel dist (train):")
print(train_df["label"].value_counts())

LABEL_ORDER = ["benign", "phishing", "malware", "other"]
le = LabelEncoder()
le.fit(LABEL_ORDER)
N_CLS = len(le.classes_)
for df in [train_df, val_df, test_df]:
    df["label_enc"] = le.transform(df["label"])
print("\nLabel mapping:", dict(zip(le.classes_, le.transform(le.classes_).tolist())))

y_train = train_df["label_enc"].values
y_val   = val_df["label_enc"].values
y_test  = test_df["label_enc"].values
benign_idx = list(le.classes_).index("benign")

In [ ]:
def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    return h.hexdigest().upper()

for fname in ["train.csv", "val.csv", "test.csv", "indonesian_threat_casestudy.csv"]:
    p = DATA_DIR / fname
    if p.exists():
        print(f"  [OK]  {fname}  ({p.stat().st_size/1024:.1f} KB, sha256={sha256_file(p)[:16]}...)")
    else:
        print(f"  [MISSING] {fname}")

### Production-readiness gate #1 — data validation

Sama seperti v4: cek konsistensi label, kebocoran domain antar split, dan cakupan panjang URL.

In [ ]:
if "is_malicious" in train_df.columns:
    mismatch = ((train_df["label"] != "benign").astype(int) != train_df["is_malicious"].astype(int))
    print(f"Label/is_malicious mismatch (train): {mismatch.mean()*100:.2f}%  ({mismatch.sum():,} rows)")

DOMAIN_COL = "registered_domain" if "registered_domain" in train_df.columns else None
if DOMAIN_COL:
    train_domains = set(train_df[DOMAIN_COL].dropna().str.lower())
    for name, df in [("val", val_df), ("test", test_df)]:
        overlap = df[DOMAIN_COL].str.lower().isin(train_domains)
        print(f"  {name}: {overlap.mean()*100:.2f}% of rows share a registered_domain with train "
              f"({overlap.sum():,}/{len(df):,})")
    TEST_OVERLAP_MASK = test_df[DOMAIN_COL].str.lower().isin(train_domains).values
else:
    TEST_OVERLAP_MASK = np.zeros(len(test_df), dtype=bool)

lens = train_df["url"].astype(str).str.len()
print(f"\nURL length (train): mean={lens.mean():.1f}  p95={lens.quantile(.95):.0f}  "
      f"p99={lens.quantile(.99):.0f}  max={lens.max()}")
print(f"  Share of URLs longer than 200 chars (would be truncated): {(lens>200).mean()*100:.2f}%")

---
## §1 — EDA & the "other" class mystery

In [ ]:
def path_depth(url):
    try:
        parsed = urlparse(url if str(url).startswith("http") else "http://" + str(url))
        return len([p for p in parsed.path.split("/") if p])
    except Exception:
        return 0

for df in [train_df, val_df, test_df]:
    df["url_len"] = df["url"].astype(str).str.len()
    df["path_depth"] = df["url"].apply(path_depth)

print("Mean URL length by class:")
print(train_df.groupby("label")["url_len"].mean().round(1))

### Investigasi kelas "other" (menjawab pertanyaan dari sesi sebelumnya)

Run v4 menunjukkan `judol_score` rata-rata untuk kelas "other" cuma 0.002 di training —
nyaris nol, sangat berbeda dari asumsi awal (OOD case study 100% judol). Sebelum menambah
fitur apa pun, kita cek dulu **secara murah** (cuma crosstab dari kolom yang sudah ada) apa
sebenarnya isi kelas "other" di training set ini.

In [ ]:
if "source" in train_df.columns:
    print("Source x Label crosstab (train) -- row-normalized (%):")
    ct = pd.crosstab(train_df["source"], train_df["label"], normalize="index") * 100
    print(ct.round(1))
    print("\nAbsolute counts:")
    print(pd.crosstab(train_df["source"], train_df["label"]))
else:
    print("'source' column not found.")

In [ ]:
print('Random sample of 15 "other"-labeled URLs from TRAIN (ground truth, not OOD):')
other_sample = train_df[train_df["label"] == "other"]["url"].sample(15, random_state=SEED)
for u in other_sample:
    print(" ", u)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, (col, xlim) in zip(axes, [("url_len", 300), ("path_depth", 15)]):
    for label, color in zip(LABEL_ORDER, ["#2ecc71", "#e74c3c", "#e67e22", "#9b59b6"]):
        data = train_df[train_df["label"] == label][col]
        ax.hist(data.clip(upper=xlim), bins=40, alpha=0.5, label=label, color=color, density=True)
    ax.set(title=f"{col} distribution by class", xlabel=col, ylabel="density")
    ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "eda_url_structure.png", dpi=150)
plt.show()

---
## §2 — Feature Engineering

Sama seperti v4: fitur reputasi eksternal (domain age, SSL, ASN) **tidak** dilatih di notebook
ini karena tidak bisa direkonstruksi dari dataset historis statis (lihat diskusi sesi sebelumnya
soal SSL/domain-age metadata) — itu masuk lapisan serving-time terpisah
(`apply_realtime_overrides()`, diekspor di §14). Leksikon judol tetap dipertahankan sebagai salah
satu cabang input (bukan lagi fitur tunggal yang diharapkan menyelesaikan semuanya sendirian) —
lihat §13 untuk investigasi klaster "other" yang lebih dalam.

In [ ]:
BRANDS = [
    "paypal", "facebook", "google", "apple", "amazon", "microsoft",
    "netflix", "instagram", "twitter", "linkedin", "dropbox", "whatsapp",
    "outlook", "office365", "icloud", "ebay", "adobe",
    "klikbca", "mandiri", "bri", "bni", "permata", "cimb", "btn", "danamon",
    "tokopedia", "shopee", "gojek", "grab", "traveloka", "dana", "ovo",
    "gopay", "linkaja", "bpjs", "pajak", "dukcapil",
]

HIGH_RISK_TLDS = {
    ".xyz", ".top", ".club", ".info", ".online", ".site", ".live",
    ".icu", ".tk", ".ml", ".ga", ".cf", ".gq", ".pw", ".cc",
    ".buzz", ".fun", ".click", ".link", ".loan", ".work", ".win",
    ".cyou", ".sbs", ".cfd", ".rest", ".bond", ".monster", ".beauty",
    ".lol", ".mom", ".quest", ".lat", ".vip",
}

SUSPICIOUS_KW = [
    "login", "signin", "verify", "secure", "account", "update",
    "confirm", "banking", "wallet", "password", "credential",
    "webscr", "cmd=_", "click", "redirect", "suspend", "limited",
    "unlock", "validate",
    "daftar", "masuk", "konfirmasi", "verifikasi", "keamanan", "blokir",
]

JUDOL_KW = [
    "slot", "gacor", "maxwin", "rtp", "jackpot", "togel", "toto",
    "judi", "taruhan", "pragmatic", "olympus", "zeus", "mahjong",
    "scatter", "spaceman", "sbobet", "rungkad", "freebet", "bonanza",
    "starlight", "gatotkaca", "wildbandito",
]

SHORTENERS = {
    "bit.ly", "tinyurl.com", "goo.gl", "t.co", "ow.ly", "is.gd", "buff.ly",
    "adf.ly", "cutt.ly", "s.id", "bit.do", "shorte.st", "rebrand.ly", "tiny.cc",
}


def levenshtein(s1, s2):
    if len(s1) < len(s2):
        s1, s2 = s2, s1
    prev = list(range(len(s2) + 1))
    for c1 in s1:
        curr = [prev[0] + 1]
        for j, c2 in enumerate(s2):
            curr.append(min(prev[j + 1] + 1, curr[-1] + 1, prev[j] + (c1 != c2)))
        prev = curr
    return prev[-1]


def brand_min_distance(domain: str, brands=BRANDS) -> int:
    domain = domain.lower()
    best = 99
    for b in brands:
        if b in domain:
            return 0
        for i in range(max(1, len(domain) - len(b) + 1)):
            d = levenshtein(domain[i:i + len(b)], b)
            if d < best:
                best = d
        best = min(best, levenshtein(domain, b))
    return best


def brand_domain_mismatch(url_lower: str, domain: str, brands=BRANDS) -> int:
    domain = domain.lower()
    for b in brands:
        if b in url_lower and b not in domain:
            return 1
    return 0


def shannon_entropy(s):
    if not s:
        return 0.0
    cnt = Counter(s)
    n = len(s)
    return -sum((v / n) * math.log2(v / n) for v in cnt.values())


def max_consecutive_consonants(s):
    vowels = set("aeiou")
    best = cur = 0
    for c in s.lower():
        if c.isalpha() and c not in vowels:
            cur += 1
            best = max(best, cur)
        elif c.isalpha():
            cur = 0
    return best


def extract_features(url: str) -> dict:
    url = str(url)
    try:
        parsed = urlparse(url if url.startswith("http") else "http://" + url)
    except Exception:
        parsed = urlparse("http://unknown")
    ext = tldextract.extract(url)
    domain = ext.domain or ""
    subdomain = ext.subdomain or ""
    suffix = ext.suffix or ""
    registered_domain = f"{domain}.{suffix}".strip(".")
    path = parsed.path or ""
    query = parsed.query or ""
    host = parsed.netloc.split(":")[0]
    path_parts = [p for p in path.split("/") if p]
    is_ip = bool(re.match(r"^\d{1,3}(\.\d{1,3}){3}$", host))
    lev_min = brand_min_distance(domain)
    url_lower = url.lower()
    judol_hits = sum(1 for kw in JUDOL_KW if kw in url_lower)

    return {
        "url_len": len(url),
        "domain_len": len(host),
        "path_len": len(path),
        "query_len": len(query),
        "path_depth": len(path_parts),
        "query_param_cnt": len(query.split("&")) if query else 0,
        "num_digits": sum(c.isdigit() for c in url),
        "num_dashes": url.count("-"),
        "num_dots": url.count("."),
        "num_slashes": url.count("/"),
        "num_special": sum(c in "@?=&%+#" for c in url),
        "num_upper": sum(c.isupper() for c in url),
        "subdomain_count": len([s for s in subdomain.split(".") if s]),
        "has_at": int("@" in url),
        "has_port": int(":" in host),
        "uses_https": int(parsed.scheme == "https"),
        "is_ip": int(is_ip),
        "tld_risk": float(f".{suffix}".lower() in HIGH_RISK_TLDS),
        "levenshtein_min": lev_min,
        "keyword_count": sum(1 for kw in SUSPICIOUS_KW if kw in url_lower),
        "entropy_domain": shannon_entropy(domain),
        "entropy_url": shannon_entropy(url),
        "has_brand_name": int(any(b in url_lower for b in BRANDS)),
        "brand_mismatch": brand_domain_mismatch(url_lower, domain),
        "ratio_digits": sum(c.isdigit() for c in url) / max(len(url), 1),
        "ratio_dashes": url.count("-") / max(len(host), 1),
        "typosquat_subs": len(re.findall(r"[0-9](?=[a-z])|[a-z](?=[0-9])", domain)),
        "has_punycode": int("xn--" in host.lower()),
        "is_shortener": int(registered_domain.lower() in SHORTENERS),
        "judol_score": judol_hits,
        "has_judol_kw": int(judol_hits > 0),
        "max_consonant_run": max_consecutive_consonants(domain),
        "vowel_ratio_domain": sum(c in "aeiou" for c in domain.lower()) / max(len(domain), 1),
        "www_prefix": int(subdomain.lower() == "www"),
        "num_www_like": int(bool(re.search(r"w{2,}\d?\.", host.lower()))),
        "ip_in_query": int(bool(re.search(r"\d{1,3}(?:\.\d{1,3}){3}", query))),
    }

In [ ]:
print("Extracting features for all splits...")
t0 = time.time()

def extract_df(df):
    return pd.DataFrame(df["url"].apply(extract_features).tolist(), index=df.index)

X_train_lex = extract_df(train_df); print(f"  train: {time.time()-t0:.1f}s")
X_val_lex   = extract_df(val_df);   print(f"  val:   {time.time()-t0:.1f}s")
X_test_lex  = extract_df(test_df);  print(f"  test:  {time.time()-t0:.1f}s")
X_ood_lex   = extract_df(ood_df);   print(f"  ood:   {time.time()-t0:.1f}s  [total]")

LEX_COLS = X_train_lex.columns.tolist()
print(f"\nEngineered features: {len(LEX_COLS)}")

In [ ]:
X_tr = X_train_lex[LEX_COLS].fillna(-1).values
X_vl = X_val_lex[LEX_COLS].fillna(-1).values
X_te = X_test_lex[LEX_COLS].fillna(-1).values
print("Raw (unscaled) lexical feature matrix shape:", X_tr.shape)

# Scaled versions -- needed for the neural-net branch in §6 (NNs are scale-sensitive,
# unlike the tree models in §3 which use X_tr/X_vl/X_te directly).
lex_scaler = StandardScaler()
X_tr_scaled = lex_scaler.fit_transform(X_tr).astype(np.float32)
X_vl_scaled = lex_scaler.transform(X_vl).astype(np.float32)
X_te_scaled = lex_scaler.transform(X_te).astype(np.float32)
X_ood_scaled = lex_scaler.transform(X_ood_lex[LEX_COLS].fillna(-1).values).astype(np.float32)

---
## §3 — Comparable Models (Classical ML Baselines)

Bagian ini **wajib ada** terlepas dari model final yang dipilih — bukan formalitas, tapi bukti
kuantitatif untuk keputusan "pakai DL atau tidak", dan XGBoost di sini tetap berguna sebagai
laporan *feature importance* untuk analis (meski bukan model yang di-deploy).

In [ ]:
def evaluate(name, y_true, y_pred, y_proba=None):
    acc  = accuracy_score(y_true, y_pred)
    f1_w = f1_score(y_true, y_pred, average="weighted")
    f1_m = f1_score(y_true, y_pred, average="macro")
    prec = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    rec  = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    cm   = confusion_matrix(y_true, y_pred, labels=list(range(N_CLS)))
    fp   = cm.sum(axis=0) - np.diag(cm)
    fn   = cm.sum(axis=1) - np.diag(cm)
    tn   = cm.sum() - (fp + fn + np.diag(cm))
    mean_fpr = (fp / (fp + tn + 1e-9)).mean()

    r = {"Model": name, "Accuracy": acc, "F1-weighted": f1_w,
         "F1-macro": f1_m, "Precision": prec, "Recall": rec, "FPR": mean_fpr}
    if y_proba is not None:
        y_bin = label_binarize(y_true, classes=list(range(N_CLS)))
        r["ROC-AUC"] = roc_auc_score(y_bin, y_proba, multi_class="ovr", average="macro")
        r["PR-AUC"]  = average_precision_score(y_bin, y_proba, average="macro")
    print(f"\n{'='*60}\n  {name}\n{'='*60}")
    print(f"  Acc: {acc:.4f}  F1-w: {f1_w:.4f}  F1-m: {f1_m:.4f}")
    print(f"  Prec: {prec:.4f}  Rec: {rec:.4f}  FPR: {mean_fpr:.4f}")
    if "PR-AUC" in r:
        print(f"  ROC-AUC: {r['ROC-AUC']:.4f}  PR-AUC: {r['PR-AUC']:.4f}")
    print(classification_report(y_true, y_pred, target_names=le.classes_, zero_division=0))
    return r

RESULTS = []

In [ ]:
# Logistic Regression
scaler_lr = StandardScaler()
X_tr_lr = scaler_lr.fit_transform(X_tr)
X_te_lr = scaler_lr.transform(X_te)
lr = LogisticRegression(max_iter=1000, C=1.0, random_state=SEED, n_jobs=-1)
lr.fit(X_tr_lr, y_train)
RESULTS.append(evaluate("Logistic Regression", y_test, lr.predict(X_te_lr), lr.predict_proba(X_te_lr)))

In [ ]:
# Naive Bayes
scaler_nb = MinMaxScaler()
nb = GaussianNB()
nb.fit(scaler_nb.fit_transform(X_tr), y_train)
RESULTS.append(evaluate("Naive Bayes", y_test, nb.predict(scaler_nb.transform(X_te)),
                        nb.predict_proba(scaler_nb.transform(X_te))))

In [ ]:
# SVM (LinearSVC + calibration for probabilities)
scaler_svm = StandardScaler()
X_tr_svm = scaler_svm.fit_transform(X_tr)
X_te_svm = scaler_svm.transform(X_te)
svm_base = LinearSVC(C=1.0, random_state=SEED, max_iter=5000)
svm = CalibratedClassifierCV(svm_base, cv=3)
svm.fit(X_tr_svm, y_train)
RESULTS.append(evaluate("SVM (LinearSVC calibrated)", y_test, svm.predict(X_te_svm), svm.predict_proba(X_te_svm)))

In [ ]:
# Decision Tree
dt = DecisionTreeClassifier(max_depth=15, min_samples_leaf=5, random_state=SEED)
dt.fit(X_tr, y_train)
RESULTS.append(evaluate("Decision Tree", y_test, dt.predict(X_te), dt.predict_proba(X_te)))

In [ ]:
# Random Forest
rf = RandomForestClassifier(n_estimators=200, max_depth=20, min_samples_leaf=3,
                             n_jobs=-1, random_state=SEED)
rf.fit(X_tr, y_train)
RESULTS.append(evaluate("Random Forest", y_test, rf.predict(X_te), rf.predict_proba(X_te)))

### XGBoost (Optuna-tuned) — the strongest classical baseline

Ini yang jadi acuan utama "seberapa bagus classical ML bisa, sebelum lompat ke DL".

In [ ]:
dtrain = xgb.DMatrix(X_tr, label=y_train)
dval   = xgb.DMatrix(X_vl, label=y_val)
dtest  = xgb.DMatrix(X_te, label=y_test)

def xgb_objective(trial):
    params = {
        "objective": "multi:softprob", "num_class": N_CLS, "eval_metric": "mlogloss",
        "eta":              trial.suggest_float("eta", 0.03, 0.3, log=True),
        "max_depth":        trial.suggest_int("max_depth", 4, 10),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "subsample":        trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "gamma":            trial.suggest_float("gamma", 0.0, 5.0),
        "lambda":           trial.suggest_float("lambda", 0.1, 10.0, log=True),
        "alpha":            trial.suggest_float("alpha", 0.0, 5.0),
        "seed": SEED, "verbosity": 0,
        "device": "cuda" if DEVICE == "cuda" else "cpu",
    }
    m = xgb.train(params, dtrain, num_boost_round=500,
                  evals=[(dval, "val")], early_stopping_rounds=30, verbose_eval=False)
    p = m.predict(dval).reshape(-1, N_CLS)
    y_bin = label_binarize(y_val, classes=list(range(N_CLS)))
    return average_precision_score(y_bin, p, average="macro")

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(xgb_objective, n_trials=60, show_progress_bar=True)
print(f"Best PR-AUC (val): {study.best_value:.4f}")
print("Best params:", study.best_params)

In [ ]:
xgb_params = dict(study.best_params)
xgb_params.update({"objective": "multi:softprob", "num_class": N_CLS,
                   "eval_metric": ["mlogloss", "merror"], "seed": SEED,
                   "device": "cuda" if DEVICE == "cuda" else "cpu"})

xgb_final = xgb.train(xgb_params, dtrain, num_boost_round=800,
                      evals=[(dval, "val")], early_stopping_rounds=30, verbose_eval=100)
xgb_final.save_model(str(OUT_DIR / "xgboost_baseline.json"))

p_xgb_val  = xgb_final.predict(dval).reshape(-1, N_CLS)
p_xgb_test = xgb_final.predict(dtest).reshape(-1, N_CLS)
pred_xgb   = p_xgb_test.argmax(1)
RESULTS.append(evaluate("XGBoost (Optuna-tuned)", y_test, pred_xgb, p_xgb_test))

In [ ]:
# Feature importance -- kept for analyst-facing explainability even though this
# isn't necessarily the final deployed model (see §7).
fi = pd.DataFrame({"feat": list(xgb_final.get_score(importance_type="gain").keys()),
                   "imp": list(xgb_final.get_score(importance_type="gain").values())})
fi = fi.sort_values("imp", ascending=True).tail(25)
fig, ax = plt.subplots(figsize=(7, 8))
ax.barh(fi["feat"], fi["imp"], color="#3498db")
ax.set(title="XGBoost — Feature Importance (gain) — for analyst explainability")
plt.tight_layout()
plt.savefig(OUT_DIR / "xgboost_feature_importance.png", dpi=150)
plt.show()

---
## §4 — TF-IDF Char N-gram + Linear Model

Baseline murah lain yang ternyata (dari run v4) jauh mengalahkan XGBoost lexical-only
(80% vs 63% akurasi standalone) — bukti kuat kalau pola substring/n-gram jauh lebih informatif
dibanding statistik ringkasan tangan-buatan. Dipertahankan sebagai baseline pembanding DAN
sebagai salah satu cabang input AfferentNet di §6.

In [ ]:
tfidf = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5),
                        max_features=4000, sublinear_tf=True)
Xtr_tfidf = tfidf.fit_transform(train_df["url"].astype(str))
Xvl_tfidf = tfidf.transform(val_df["url"].astype(str))
Xte_tfidf = tfidf.transform(test_df["url"].astype(str))
Xood_tfidf = tfidf.transform(ood_df["url"].astype(str))

sgd = SGDClassifier(loss="log_loss", alpha=1e-5, max_iter=50,
                    random_state=SEED, n_jobs=-1, class_weight="balanced")
sgd.fit(Xtr_tfidf, y_train)

p_sgd_val  = sgd.predict_proba(Xvl_tfidf)
p_sgd_test = sgd.predict_proba(Xte_tfidf)
p_sgd_ood  = sgd.predict_proba(Xood_tfidf)
pred_sgd   = p_sgd_test.argmax(1)
RESULTS.append(evaluate("TF-IDF + SGD (char n-grams)", y_test, pred_sgd, p_sgd_test))

with open(OUT_DIR / "tfidf_vectorizer.pkl", "wb") as f: pickle.dump(tfidf, f)
with open(OUT_DIR / "sgd_baseline.pkl", "wb") as f: pickle.dump(sgd, f)

---
## §5 — Deep Learning: Character Encoder Ablation

Run v4 kasih hasil berlawanan dengan asumsi awal: CNN-BiLSTM (arsitektur lama) justru menang
tipis atas CNN+Self-Attention di satu run itu, setelah run sebelumnya (v3-as-run) bilang
sebaliknya. Daripada menebak lagi, kita uji ulang keduanya sebagai classifier karakter berdiri
sendiri di run ini juga, dan yang menang dipakai sebagai cabang karakter di AfferentNet (§6).

In [ ]:
URL_CHARS = (
    "abcdefghijklmnopqrstuvwxyz"
    "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    "0123456789"
    r".-_~:/?#[]@!$&'()*+,;=%"
)
CHAR2IDX  = {c: i + 2 for i, c in enumerate(URL_CHARS)}  # 0=PAD, 1=UNK
VOCAB_SIZE = len(CHAR2IDX) + 2
MAX_LEN    = 200

def url_to_tensor(url):
    ids = [CHAR2IDX.get(c, 1) for c in str(url)[:MAX_LEN]]
    ids += [0] * (MAX_LEN - len(ids))
    return torch.tensor(ids, dtype=torch.long)

class URLDataset(Dataset):
    """Simple (char_ids, label) dataset -- used only for the standalone char-encoder ablation."""
    def __init__(self, urls, labels):
        self.urls = urls.reset_index(drop=True) if hasattr(urls, "reset_index") else pd.Series(list(urls)).reset_index(drop=True)
        self.labels = labels
    def __len__(self): return len(self.urls)
    def __getitem__(self, i): return url_to_tensor(self.urls[i]), int(self.labels[i])

BS = 256
def mk_loader(ds, shuffle):
    return DataLoader(ds, batch_size=BS, shuffle=shuffle, num_workers=2, pin_memory=DEVICE == "cuda")

ablation_train_loader = mk_loader(URLDataset(train_df["url"], y_train), True)
ablation_val_loader   = mk_loader(URLDataset(val_df["url"], y_val), False)
ablation_test_loader  = mk_loader(URLDataset(test_df["url"], y_test), False)
print(f"Vocab size: {VOCAB_SIZE}  Max URL length: {MAX_LEN}")

In [ ]:
class CharEncoderBiLSTM(nn.Module):
    """Char CNN (local patterns, e.g. l->1) + BiLSTM + attention pooling (global context).
    Returns a pooled representation -- reused standalone here AND as AfferentNet's char branch."""
    def __init__(self, vocab=VOCAB_SIZE, emb=64, filters=128, kernels=(3, 5, 7),
                 lstm_h=128, lstm_l=2, drop=0.3):
        super().__init__()
        self.embed = nn.Embedding(vocab, emb, padding_idx=0)
        self.convs = nn.ModuleList([
            nn.Sequential(nn.Conv1d(emb, filters, k, padding=k // 2), nn.GELU(), nn.Dropout(drop / 2))
            for k in kernels])
        cnn_dim = filters * len(kernels)
        self.bilstm = nn.LSTM(cnn_dim, lstm_h, lstm_l, bidirectional=True, batch_first=True,
                              dropout=drop if lstm_l > 1 else 0.0)
        self.attn = nn.Linear(lstm_h * 2, 1)
        self.out_dim = lstm_h * 2

    def forward(self, x):
        e = self.embed(x)
        ci = e.permute(0, 2, 1)
        c = torch.cat([cv(ci).permute(0, 2, 1) for cv in self.convs], -1)
        out, _ = self.bilstm(c)
        w = torch.softmax(self.attn(out), 1)
        return (out * w).sum(1)


class CharEncoderAttn(nn.Module):
    """Char CNN + Transformer self-attention (no recurrence) -- global context in parallel."""
    def __init__(self, vocab=VOCAB_SIZE, emb=64, filters=128, kernels=(3, 5, 7),
                 d_model=256, nhead=4, n_layers=2, drop=0.3, max_len=MAX_LEN):
        super().__init__()
        self.embed = nn.Embedding(vocab, emb, padding_idx=0)
        self.pos = nn.Parameter(torch.zeros(1, max_len, emb))
        nn.init.trunc_normal_(self.pos, std=0.02)
        self.convs = nn.ModuleList([
            nn.Sequential(nn.Conv1d(emb, filters, k, padding=k // 2), nn.GELU(), nn.Dropout(drop / 2))
            for k in kernels])
        cnn_dim = filters * len(kernels)
        self.proj = nn.Linear(cnn_dim, d_model)
        enc_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=d_model * 2,
                                               dropout=drop, activation="gelu", batch_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.attn_pool = nn.Linear(d_model, 1)
        self.out_dim = d_model

    def forward(self, x):
        pad_mask = (x == 0)
        e = self.embed(x) + self.pos[:, :x.size(1), :]
        ci = e.permute(0, 2, 1)
        c = torch.cat([cv(ci).permute(0, 2, 1) for cv in self.convs], -1)
        h = self.proj(c)
        h = self.encoder(h, src_key_padding_mask=pad_mask)
        w = self.attn_pool(h).masked_fill(pad_mask.unsqueeze(-1), -1e4)
        w = torch.softmax(w, 1)
        return (h * w).sum(1)


class CharClassifier(nn.Module):
    """Thin classification head on top of a char encoder -- for the standalone ablation only."""
    def __init__(self, encoder, n_cls=N_CLS, drop=0.3):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Sequential(nn.Linear(encoder.out_dim, 128), nn.GELU(),
                                  nn.Dropout(drop), nn.Linear(128, n_cls))
    def forward(self, x):
        return self.head(self.encoder(x))

In [ ]:
def train_simple(model, train_loader, val_loader, epochs=14, lr=3e-4, patience=5):
    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    crit  = nn.CrossEntropyLoss(label_smoothing=0.05)
    best_vl, best_state, no_improve = float("inf"), None, 0
    for ep in range(1, epochs + 1):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        model.eval(); vl = 0.0; correct = 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                logits = model(xb)
                vl += nn.functional.cross_entropy(logits, yb).item() * len(xb)
                correct += (logits.argmax(1) == yb).sum().item()
        vl /= len(val_loader.dataset)
        va = correct / len(val_loader.dataset)
        sched.step()
        if vl < best_vl - 1e-5:
            best_vl = vl; best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}; no_improve = 0
        else:
            no_improve += 1
        print(f"  Epoch {ep:02d}/{epochs} | ValLoss {vl:.4f} | ValAcc {va:.4f}" + (" [best]" if no_improve == 0 else ""))
        if no_improve >= patience:
            print(f"  Early stopping (no val improvement for {patience} epochs)")
            break
    model.load_state_dict(best_state)
    return best_vl

def infer_simple(model, loader):
    model.eval(); probas = []
    with torch.no_grad():
        for xb, _ in loader:
            xb = xb.to(DEVICE)
            probas.append(torch.softmax(model(xb), 1).cpu().numpy())
    probas = np.concatenate(probas, 0)
    return probas, probas.argmax(1)

print("Training char encoder (BiLSTM)...")
enc_bilstm = CharEncoderBiLSTM().to(DEVICE)
clf_bilstm = CharClassifier(enc_bilstm).to(DEVICE)
train_simple(clf_bilstm, ablation_train_loader, ablation_val_loader, epochs=14)
p_bilstm_val, _ = infer_simple(clf_bilstm, ablation_val_loader)
p_bilstm_test, pred_bilstm = infer_simple(clf_bilstm, ablation_test_loader)
RESULTS.append(evaluate("Char Encoder: BiLSTM (ablation)", y_test, pred_bilstm, p_bilstm_test))

In [ ]:
print("Training char encoder (Self-Attention)...")
enc_attn = CharEncoderAttn().to(DEVICE)
clf_attn = CharClassifier(enc_attn).to(DEVICE)
train_simple(clf_attn, ablation_train_loader, ablation_val_loader, epochs=14)
p_attn_val, _ = infer_simple(clf_attn, ablation_val_loader)
p_attn_test, pred_attn = infer_simple(clf_attn, ablation_test_loader)
RESULTS.append(evaluate("Char Encoder: Self-Attention (ablation)", y_test, pred_attn, p_attn_test))

In [ ]:
acc_bilstm = accuracy_score(y_test, pred_bilstm)
acc_attn   = accuracy_score(y_test, pred_attn)
if acc_bilstm >= acc_attn:
    print(f"Winner: BiLSTM ({acc_bilstm:.4f} vs {acc_attn:.4f}) -- used as AfferentNet's char branch.")
    WINNING_ENCODER_CLS = CharEncoderBiLSTM
else:
    print(f"Winner: Self-Attention ({acc_attn:.4f} vs {acc_bilstm:.4f}) -- used as AfferentNet's char branch.")
    WINNING_ENCODER_CLS = CharEncoderAttn

---
## §6 — AfferentNet: Unified Multi-Branch Production Model

Satu jaringan, dilatih end-to-end, menyatukan 3 sinyal yang sebelumnya dikombinasikan lewat
stacker terpisah di v4 (yang ternyata rawan sirkularitas validation-set):

- **Cabang karakter** — encoder pemenang dari §5 (pola lokal + konteks global dari urutan karakter)
- **Cabang lexical** — MLP kecil di atas 36 fitur tangan-buatan (brand mismatch, judol score, TLD
  risk, entropy, dst dari §2)
- **Cabang TF-IDF** — proyeksi dense dari fitur char n-gram §4

Ketiganya digabung (concatenate) lalu diklasifikasi bersama — idealnya mendapat manfaat
"menggabungkan 3 sinyal" seperti stacker v4, tapi dioptimasi bersama (joint training) dan
jauh lebih sederhana untuk di-deploy: **satu model, satu forward pass**, tanpa kalibrasi
dan stacking berlapis yang rawan bug seperti di v4.

In [ ]:
class AfferentDataset(Dataset):
    """Returns (char_ids, lex_feats, tfidf_feats, label) -- all 3 AfferentNet input branches."""
    def __init__(self, urls, lex_arr, tfidf_mat, labels):
        self.urls = urls.reset_index(drop=True) if hasattr(urls, "reset_index") else pd.Series(list(urls)).reset_index(drop=True)
        self.lex_arr = np.asarray(lex_arr, dtype=np.float32)
        self.tfidf_mat = tfidf_mat.tocsr()
        self.labels = labels
    def __len__(self):
        return len(self.urls)
    def __getitem__(self, i):
        char_ids = url_to_tensor(self.urls[i])
        lex = torch.from_numpy(self.lex_arr[i])
        tfidf_row = torch.from_numpy(self.tfidf_mat[i].toarray().ravel().astype(np.float32))
        return char_ids, lex, tfidf_row, int(self.labels[i])

def mk_loader2(ds, shuffle):
    return DataLoader(ds, batch_size=BS, shuffle=shuffle, num_workers=2, pin_memory=DEVICE == "cuda")

afferent_train_loader = mk_loader2(AfferentDataset(train_df["url"], X_tr_scaled, Xtr_tfidf, y_train), True)
afferent_val_loader   = mk_loader2(AfferentDataset(val_df["url"],   X_vl_scaled, Xvl_tfidf, y_val), False)
afferent_test_loader  = mk_loader2(AfferentDataset(test_df["url"],  X_te_scaled, Xte_tfidf, y_test), False)
afferent_ood_loader   = mk_loader2(AfferentDataset(ood_df["url"],   X_ood_scaled, Xood_tfidf, np.zeros(len(ood_df), int)), False)
print("AfferentNet data loaders ready.")

In [ ]:
class AfferentNet(nn.Module):
    def __init__(self, char_encoder, n_lex, n_tfidf, n_cls=N_CLS,
                 lex_hidden=64, tfidf_proj_dim=128, fusion_hidden=256, drop=0.35):
        super().__init__()
        self.char_encoder = char_encoder
        char_out_dim = char_encoder.out_dim
        self.lex_mlp = nn.Sequential(
            nn.Linear(n_lex, lex_hidden), nn.GELU(), nn.Dropout(drop),
            nn.Linear(lex_hidden, lex_hidden), nn.GELU())
        self.tfidf_proj = nn.Sequential(
            nn.Linear(n_tfidf, tfidf_proj_dim), nn.GELU(), nn.Dropout(drop))
        fusion_in = char_out_dim + lex_hidden + tfidf_proj_dim
        self.fusion = nn.Sequential(
            nn.Linear(fusion_in, fusion_hidden), nn.GELU(), nn.Dropout(drop),
            nn.Linear(fusion_hidden, n_cls))

    def forward(self, char_ids, lex_feats, tfidf_feats):
        c = self.char_encoder(char_ids)
        l = self.lex_mlp(lex_feats)
        t = self.tfidf_proj(tfidf_feats)
        return self.fusion(torch.cat([c, l, t], dim=1))

winner_encoder = WINNING_ENCODER_CLS().to(DEVICE)
afferent_net = AfferentNet(winner_encoder, n_lex=len(LEX_COLS), n_tfidf=Xtr_tfidf.shape[1]).to(DEVICE)
total_params = sum(p.numel() for p in afferent_net.parameters())
print(f"AfferentNet parameters: {total_params:,}  (char branch: {WINNING_ENCODER_CLS.__name__})")

In [ ]:
def train_afferentnet(model, train_loader, val_loader, epochs=20, lr=2e-4, patience=6):
    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    crit  = nn.CrossEntropyLoss(label_smoothing=0.05)
    best_vl, best_state, no_improve = float("inf"), None, 0
    tr_hist, vl_hist = [], []
    for ep in range(1, epochs + 1):
        model.train(); tl = 0.0
        for cb, lb, tb, yb in train_loader:
            cb, lb, tb, yb = cb.to(DEVICE), lb.to(DEVICE), tb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(cb, lb, tb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tl += loss.item() * len(cb)
        model.eval(); vl = 0.0; correct = 0
        with torch.no_grad():
            for cb, lb, tb, yb in val_loader:
                cb, lb, tb, yb = cb.to(DEVICE), lb.to(DEVICE), tb.to(DEVICE), yb.to(DEVICE)
                logits = model(cb, lb, tb)
                vl += crit(logits, yb).item() * len(cb)
                correct += (logits.argmax(1) == yb).sum().item()
        tl /= len(train_loader.dataset); vl /= len(val_loader.dataset)
        va = correct / len(val_loader.dataset)
        tr_hist.append(tl); vl_hist.append(vl)
        sched.step()
        if vl < best_vl - 1e-5:
            best_vl = vl; best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}; no_improve = 0
        else:
            no_improve += 1
        print(f"  Epoch {ep:02d}/{epochs} | TrLoss {tl:.4f} | ValLoss {vl:.4f} | ValAcc {va:.4f}"
              + (" [best]" if no_improve == 0 else ""))
        if no_improve >= patience:
            print(f"  Early stopping (no val improvement for {patience} epochs)")
            break
    model.load_state_dict(best_state)
    return tr_hist, vl_hist

def afferentnet_infer(model, loader):
    model.eval(); probas = []
    with torch.no_grad():
        for cb, lb, tb, _ in loader:
            cb, lb, tb = cb.to(DEVICE), lb.to(DEVICE), tb.to(DEVICE)
            probas.append(torch.softmax(model(cb, lb, tb), 1).cpu().numpy())
    probas = np.concatenate(probas, 0)
    return probas, probas.argmax(1)

print("Training AfferentNet (joint, end-to-end)...")
tr_hist, vl_hist = train_afferentnet(afferent_net, afferent_train_loader, afferent_val_loader, epochs=22)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(tr_hist, label="Train Loss", color="#3498db")
ax.plot(vl_hist, label="Val Loss", color="#e74c3c")
ax.set(title="AfferentNet training curve", xlabel="epoch", ylabel="loss")
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "afferentnet_training_curve.png", dpi=150)
plt.show()

p_afferent_val, _ = afferentnet_infer(afferent_net, afferent_val_loader)
p_afferent_test, pred_afferent = afferentnet_infer(afferent_net, afferent_test_loader)
RESULTS.append(evaluate("AfferentNet (unified multi-branch DL)", y_test, pred_afferent, p_afferent_test))

torch.save(afferent_net.state_dict(), OUT_DIR / "afferentnet.pt")
with open(OUT_DIR / "afferentnet_arch.json", "w") as f:
    json.dump({"char_encoder": WINNING_ENCODER_CLS.__name__, "n_lex": len(LEX_COLS),
              "n_tfidf": Xtr_tfidf.shape[1], "n_cls": N_CLS, "vocab": VOCAB_SIZE,
              "max_len": MAX_LEN}, f, indent=2)
print("AfferentNet saved.")

---
## §7 — Model Comparison & Final Selection (data-driven, not assumed)

Semua kandidat "serius" (bukan baseline lemah seperti LR/NB/SVM/DT yang dari 2 run sebelumnya
konsisten di bawah 65%) dikumpulkan di satu peta, dan model final dipilih murni berdasarkan
F1-weighted tertinggi di test set — sesuai permintaan: kalau DL menang, DL yang dipakai penuh,
tanpa tier/cascade tambahan.

In [ ]:
res_df = pd.DataFrame(RESULTS).set_index("Model").round(4)
print("=" * 70)
print("FULL MODEL COMPARISON -- TEST SET (classical baselines + DL candidates)")
print("=" * 70)
print(res_df.to_string())

plot_cols = [c for c in ["Accuracy", "F1-weighted", "PR-AUC", "FPR"] if c in res_df.columns]
fig, ax = plt.subplots(figsize=(12, len(res_df) * 0.5 + 2))
sns.heatmap(res_df[plot_cols].astype(float), annot=True, fmt=".4f", cmap="RdYlGn",
           cbar_kws={"label": "score"}, ax=ax)
ax.set(title="All Models — Test Set Comparison")
plt.tight_layout()
plt.savefig(OUT_DIR / "all_models_heatmap.png", dpi=150)
plt.show()

In [ ]:
# Prediction/probability lookup for the serious contenders only (weak classical
# baselines are shown above for completeness but were never going to be deployed).
PRED_MAP = {
    "XGBoost (Optuna-tuned)":                 (pred_xgb, p_xgb_test),
    "TF-IDF + SGD (char n-grams)":             (pred_sgd, p_sgd_test),
    "Char Encoder: BiLSTM (ablation)":         (pred_bilstm, p_bilstm_test),
    "Char Encoder: Self-Attention (ablation)": (pred_attn, p_attn_test),
    "AfferentNet (unified multi-branch DL)":   (pred_afferent, p_afferent_test),
}
PROBA_VAL_MAP = {
    "XGBoost (Optuna-tuned)":                 p_xgb_val,
    "TF-IDF + SGD (char n-grams)":             p_sgd_val,
    "Char Encoder: BiLSTM (ablation)":         p_bilstm_val,
    "Char Encoder: Self-Attention (ablation)": p_attn_val,
    "AfferentNet (unified multi-branch DL)":   p_afferent_val,
}

contender_names = list(PRED_MAP.keys())
FINAL_MODEL_NAME = res_df.loc[contender_names, "F1-weighted"].idxmax()
final_pred, final_p = PRED_MAP[FINAL_MODEL_NAME]
final_p_val = PROBA_VAL_MAP[FINAL_MODEL_NAME]

print(f"\n>>> SELECTED PRODUCTION MODEL: {FINAL_MODEL_NAME} <<<")
print(res_df.loc[FINAL_MODEL_NAME].to_string())

if FINAL_MODEL_NAME != "AfferentNet (unified multi-branch DL)":
    print(f"\nNOTE: {FINAL_MODEL_NAME} outperformed AfferentNet on this run.")
    print("Sections from here on (adversarial testing, latency, export) are written assuming")
    print("AfferentNet is deployed, since it is the richest/most-likely winner across runs so far.")
    print("If a different model actually won here, those sections need re-adapting before shipping.")

---
## §8 — Calibration & Threshold Governance (circularity bug from v4, fixed)

v4 fit kalibrasi Tier-1/Tier-2 DAN stacker DAN pemilihan threshold semuanya di validation set
yang sama — hasilnya threshold yang kelihatan oke di val (gap 0.87pp dari full stacker) jebol
parah di test (gap 12.3pp). Perbaikannya di v5: validation set **dibelah dua** — satu untuk
fit kalibrasi, satu lagi (belum pernah disentuh kalibrasi) murni untuk pilih threshold.

In [ ]:
def fit_isotonic_calibrators(y_ref, p_ref, n_classes):
    calibrators = []
    y_bin = label_binarize(y_ref, classes=list(range(n_classes)))
    for c in range(n_classes):
        ir = IsotonicRegression(out_of_bounds="clip")
        ir.fit(p_ref[:, c], y_bin[:, c])
        calibrators.append(ir)
    return calibrators

def apply_isotonic(calibrators, p):
    out = np.column_stack([cal.predict(p[:, c]) for c, cal in enumerate(calibrators)])
    out = out / out.sum(axis=1, keepdims=True).clip(min=1e-9)
    return out

val_idx_calib, val_idx_thresh = train_test_split(
    np.arange(len(val_df)), test_size=0.5, stratify=y_val, random_state=SEED)
print(f"Validation split for calibration: {len(val_idx_calib):,} rows")
print(f"Validation split for threshold selection (held out from calibration): {len(val_idx_thresh):,} rows")

cal_final = fit_isotonic_calibrators(y_val[val_idx_calib], final_p_val[val_idx_calib], N_CLS)
final_p_test_cal = apply_isotonic(cal_final, final_p)
RESULTS.append(evaluate(f"{FINAL_MODEL_NAME} [calibrated]", y_test, final_p_test_cal.argmax(1), final_p_test_cal))

In [ ]:
def find_threshold_for_fpr(y_true_mal, p_mal, target_fpr):
    fpr, tpr, thr = roc_curve(y_true_mal, p_mal)
    valid = fpr <= target_fpr
    idx = np.where(valid)[0][-1] if valid.any() else 0
    return float(thr[idx]), float(fpr[idx]), float(tpr[idx])

p_val_thresh_cal = apply_isotonic(cal_final, final_p_val[val_idx_thresh])
y_val_thresh = y_val[val_idx_thresh]
if "is_malicious" in val_df.columns:
    y_val_thresh_mal = val_df["is_malicious"].values[val_idx_thresh].astype(int)
else:
    y_val_thresh_mal = (y_val_thresh != benign_idx).astype(int)

pmal_val_thresh = 1 - p_val_thresh_cal[:, benign_idx]
thr_block, fpr_block, tpr_block = find_threshold_for_fpr(y_val_thresh_mal, pmal_val_thresh, target_fpr=0.02)
thr_review, fpr_review, tpr_review = find_threshold_for_fpr(y_val_thresh_mal, pmal_val_thresh, target_fpr=0.10)

print(f"Block-auto threshold  (target FPR<=2%):  P(malicious) >= {thr_block:.3f}  "
      f"(held-out val FPR={fpr_block:.4f}, recall={tpr_block:.4f})")
print(f"Block-review threshold (target FPR<=10%): P(malicious) >= {thr_review:.3f}  "
      f"(held-out val FPR={fpr_review:.4f}, recall={tpr_review:.4f})")

# Verify on TEST (fully independent of both calibration-fit and threshold-selection data)
pmal_test = 1 - final_p_test_cal[:, benign_idx]
y_test_mal = test_df["is_malicious"].values.astype(int) if "is_malicious" in test_df.columns \
    else (y_test != benign_idx).astype(int)
pred_mal_test = (pmal_test >= thr_block).astype(int)
test_fpr = ((pred_mal_test == 1) & (y_test_mal == 0)).sum() / max((y_test_mal == 0).sum(), 1)
test_recall = ((pred_mal_test == 1) & (y_test_mal == 1)).sum() / max((y_test_mal == 1).sum(), 1)
print(f"\nOn TEST with block_auto threshold: FPR={test_fpr:.4f}  Recall(malicious)={test_recall:.4f}")
print("(if this held-out-test gap looks small and consistent with the held-out-val numbers above,")
print(" the circularity bug from v4 is fixed -- the threshold now genuinely generalizes.)")

ACTION_THRESHOLDS = {"block_auto": thr_block, "block_review": thr_review, "allow_log": 0.0}

---
## §9 — Latency Benchmark (single model, CPU — realistic online-serving estimate)

In [ ]:
def predict_afferentnet(urls, model=afferent_net, batch_size=256, device=DEVICE):
    """Reference serving-path implementation: raw URL strings in, calibrated-ready softmax out.
    Used for latency benchmarking, adversarial testing, and OOD scoring below."""
    model.eval()
    urls = list(urls)
    all_p = []
    with torch.no_grad():
        for i in range(0, len(urls), batch_size):
            batch = urls[i:i + batch_size]
            lex_feats = pd.DataFrame([extract_features(u) for u in batch])[LEX_COLS].fillna(-1).values
            lex_feats = lex_scaler.transform(lex_feats).astype(np.float32)
            tfidf_feats = tfidf.transform(batch).toarray().astype(np.float32)
            char_ids = torch.stack([url_to_tensor(u) for u in batch]).to(device)
            lb = torch.from_numpy(lex_feats).to(device)
            tb = torch.from_numpy(tfidf_feats).to(device)
            p = torch.softmax(model(char_ids, lb, tb), dim=1).cpu().numpy()
            all_p.append(p)
    return np.concatenate(all_p, 0)

def bench_afferentnet_cpu(n=300):
    afferent_net.to("cpu")
    sample_urls = test_df["url"].astype(str).values[:n]
    times = []
    for u in sample_urls:
        t0 = time.perf_counter()
        predict_afferentnet([u], model=afferent_net, device="cpu")
        times.append((time.perf_counter() - t0) * 1000)
    afferent_net.to(DEVICE)
    return np.array(times)

def pct(a, p): return float(np.percentile(a, p))

if FINAL_MODEL_NAME == "AfferentNet (unified multi-branch DL)":
    lat_times = bench_afferentnet_cpu()
    print("Latency benchmark -- single URL, CPU, end-to-end (feature extraction + forward pass):")
    print(f"  p50={pct(lat_times,50):.2f}ms  p95={pct(lat_times,95):.2f}ms  p99={pct(lat_times,99):.2f}ms")
    print(f"\n  v3 target -- full path p95 < 200ms -> {'PASS' if pct(lat_times,95) < 200 else 'FAIL'}")
else:
    print(f"Skipping AfferentNet-specific latency bench -- {FINAL_MODEL_NAME} won instead; "
          f"benchmark that model's inference path separately before shipping.")

---
## §10 — Domain-Leakage Check & Tier-0 Blocklist/Allowlist Simulation

In [ ]:
print(f"Test rows sharing a registered_domain with train: {TEST_OVERLAP_MASK.mean()*100:.2f}%")
novel_mask = ~TEST_OVERLAP_MASK
if novel_mask.sum() > 20:
    acc_novel = accuracy_score(y_test[novel_mask], final_pred[novel_mask])
    acc_all   = accuracy_score(y_test, final_pred)
    print(f"Stricter generalization check -- {FINAL_MODEL_NAME}: all={acc_all:.4f}  "
          f"novel-domain-only={acc_novel:.4f}  (n_novel={novel_mask.sum():,})")
else:
    print("Near-zero domain overlap (dataset's 'zero-leakage' claim holds) -- "
          "cannot compute a meaningfully different novel-domain-only subset.")

if DOMAIN_COL:
    dom_label_purity = train_df.groupby(DOMAIN_COL)["label"].agg(lambda s: s.value_counts(normalize=True).iloc[0])
    dom_majority_label = train_df.groupby(DOMAIN_COL)["label"].agg(lambda s: s.value_counts().index[0])
    CACHE = dom_majority_label[dom_label_purity >= 0.95].to_dict()
    test_domains_lower = test_df[DOMAIN_COL].str.lower()
    cache_hit = test_domains_lower.isin(CACHE.keys()).values
    print(f"\nTier-0 cache size: {len(CACHE):,} domains  |  HIT rate on test: {cache_hit.mean()*100:.1f}%")
    if cache_hit.any():
        cache_pred_enc = le.transform(test_domains_lower[cache_hit].map(CACHE))
        print(f"  Accuracy of Tier-0 cache verdicts alone: {accuracy_score(y_test[cache_hit], cache_pred_enc):.4f}")
    print("  (0% hit rate is EXPECTED here given the dataset's zero-leakage split -- this doesn't")
    print("   mean Tier-0 is useless in production, just that this offline test set can't show its")
    print("   benefit: real traffic WILL revisit domains that build up reputation history over time.)")
else:
    CACHE = {}
    print("registered_domain column not available -- Tier-0 simulation skipped.")

---
## §11 — Adversarial Robustness Testing (new — this gap was never tested before)

Classifier berbasis lexical/karakter terkenal gampang dikelabui dengan perturbasi murah
(homoglyph, subdomain tambahan, noise di query string). Kita uji langsung: ambil URL malicious
yang BENAR diklasifikasi, beri perturbasi kecil, lihat berapa persen "lolos" jadi diprediksi
benign (evasion rate).

In [ ]:
def perturb_homoglyph(url):
    subs = {"o": "0", "l": "1", "e": "3", "a": "4", "s": "5", "i": "1"}
    chars = list(url)
    idxs = [i for i, c in enumerate(chars) if c.lower() in subs]
    if not idxs:
        return url
    rng = np.random.RandomState(abs(hash(url)) % (2**32))
    rng.shuffle(idxs)
    n_sub = max(1, len(idxs) // 10)
    for i in idxs[:n_sub]:
        chars[i] = subs[chars[i].lower()]
    return "".join(chars)

def perturb_add_subdomain(url):
    return url.replace("://", "://cdn-static.", 1) if "://" in url else "cdn-static." + url

def perturb_add_query_noise(url):
    rng = np.random.RandomState(abs(hash(url)) % (2**32))
    sep = "&" if "?" in url else "?"
    noise = "".join(rng.choice(list("abcdefghij0123456789"), 8))
    return f"{url}{sep}ref={noise}"

def perturb_case_randomize(url):
    rng = np.random.RandomState(abs(hash(url)) % (2**32))
    return "".join(c.upper() if rng.rand() < 0.3 else c for c in url)

PERTURBATIONS = {
    "homoglyph (o->0, l->1, ...)": perturb_homoglyph,
    "add benign-looking subdomain": perturb_add_subdomain,
    "query string noise": perturb_add_query_noise,
    "random case": perturb_case_randomize,
}

In [ ]:
if FINAL_MODEL_NAME == "AfferentNet (unified multi-branch DL)":
    mal_mask = (y_test != benign_idx) & (final_pred == y_test)
    n_sample = min(500, int(mal_mask.sum()))
    sample_idx = np.random.RandomState(SEED).choice(np.where(mal_mask)[0], size=n_sample, replace=False)
    sample_urls = test_df["url"].astype(str).values[sample_idx]

    print(f"Adversarial robustness test on {n_sample} correctly-detected malicious URLs:\n")
    for name, fn in PERTURBATIONS.items():
        perturbed = [fn(u) for u in sample_urls]
        p_pert = predict_afferentnet(perturbed)
        pred_pert = p_pert.argmax(1)
        evasion_rate = (pred_pert == benign_idx).mean()
        flag = "  <-- CONCERNING" if evasion_rate > 0.15 else ""
        print(f"  [{name:32s}] evasion rate (now predicted benign): {evasion_rate*100:5.1f}%{flag}")
    print("\nEvasion rate >15% on any single cheap perturbation should block auto-block-only")
    print("deployment until addressed (e.g. adversarial training / data augmentation with these")
    print("exact perturbation types added to the training set).")
else:
    print(f"Skipping AfferentNet-specific adversarial test -- re-adapt for {FINAL_MODEL_NAME}.")

---
## §12 — Segmented Evaluation & Statistical Significance

In [ ]:
print(f"Per-class Evaluation -- {FINAL_MODEL_NAME}:")
y_bin = label_binarize(y_test, classes=list(range(N_CLS)))
for i, cls in enumerate(le.classes_):
    pr_auc = average_precision_score(y_bin[:, i], final_p[:, i])
    fp_i = ((final_pred == i) & (y_test != i)).sum()
    tn_i = ((final_pred != i) & (y_test != i)).sum()
    fpr_i = fp_i / (fp_i + tn_i + 1e-9)
    n_cls_i = (y_test == i).sum()
    print(f"  [{cls:8s}] n={n_cls_i:5,}  PR-AUC={pr_auc:.4f}  FPR={fpr_i:.4f}")

if "source" in test_df.columns:
    print("\nPer-source F1:")
    for src in sorted(test_df["source"].unique()):
        mask = test_df["source"].values == src
        if mask.sum() < 10:
            continue
        f1 = f1_score(y_test[mask], final_pred[mask], average="weighted", zero_division=0)
        print(f"  [{src:30s}]  n={mask.sum():6,}  F1={f1:.4f}")

In [ ]:
def mcnemar_report(name_a, pred_a, name_b, pred_b):
    c_a = (pred_a == y_test); c_b = (pred_b == y_test)
    n11 = (c_a & c_b).sum(); n10 = (c_a & ~c_b).sum()
    n01 = (~c_a & c_b).sum(); n00 = (~c_a & ~c_b).sum()
    result = mcnemar_test([[n11, n10], [n01, n00]], exact=False, correction=True)
    print(f"McNemar's Test -- {name_a} vs {name_b}")
    print(f"  Both correct: {n11:,}   Only {name_a}: {n10:,}   Only {name_b}: {n01:,}   Both wrong: {n00:,}")
    print(f"  Chi2: {result.statistic:.4f}  p-value: {result.pvalue:.6f}  "
          f"-> {'SIGNIFICANT' if result.pvalue < 0.05 else 'not significant'} (p<0.05)\n")

mcnemar_report(FINAL_MODEL_NAME.split(' (')[0], final_pred, "XGBoost", pred_xgb)
mcnemar_report(FINAL_MODEL_NAME.split(' (')[0], final_pred, "TF-IDF+SGD", pred_sgd)

---
## §13 — OOD Evaluation & "other"-class Cluster Investigation

Selain metrik zero-shot biasa, bagian ini juga menjawab pertanyaan yang masih menggantung dari
sesi sebelumnya: **isi kelas "other" di training itu sebenarnya apa** (karena ternyata bukan
didominasi judol, lihat §1).

In [ ]:
if FINAL_MODEL_NAME == "AfferentNet (unified multi-branch DL)":
    p_ood = predict_afferentnet(ood_df["url"].astype(str).tolist())
else:
    p_ood = None
    print(f"Re-adapt OOD scoring for {FINAL_MODEL_NAME} before using the cells below.")

if p_ood is not None:
    p_ood_cal = apply_isotonic(cal_final, p_ood)
    pred_ood  = p_ood_cal.argmax(1)
    conf_ood  = p_ood_cal.max(1)
    label_ood = le.inverse_transform(pred_ood)
    print("OOD predicted class distribution:")
    print(pd.Series(label_ood).value_counts())

In [ ]:
def wilson_ci(k, n, z=1.96):
    if n == 0:
        return (0.0, 0.0)
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    margin = z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return (max(0.0, center - margin), min(1.0, center + margin))

if p_ood is not None and "label" in ood_df.columns:
    print("Per-class OOD recall (95% Wilson CI):")
    for cls in le.classes_:
        mask = ood_df["label"].values == cls
        n_c = int(mask.sum())
        if n_c == 0:
            continue
        cls_idx = list(le.classes_).index(cls)
        k_c = int((pred_ood[mask] == cls_idx).sum())
        lo, hi = wilson_ci(k_c, n_c)
        print(f"  [{cls:8s}] n={n_c:4d}  recall={k_c/n_c:.3f}  CI [{lo:.3f},{hi:.3f}]"
              + ("   (small-n: read with caution)" if n_c < 30 else ""))

    if "is_malicious" in ood_df.columns:
        y_ood_bin = ood_df["is_malicious"].values.astype(int)
        detected = (pred_ood != benign_idx).astype(int)
        print(f"\nOOD binary detection rate: {(detected==y_ood_bin).mean():.4f}")
        print(f"OOD threat recall: {((detected==1)&(y_ood_bin==1)).sum()/max(y_ood_bin.sum(),1):.4f}")
        print("v4 baseline for reference: detection 0.66, threat recall 0.62, 'other' recall 0.000")

### Investigasi klaster "other" (training set)

Daripada menebak lagi, kita cluster URL kelas "other" di training pakai TF-IDF + KMeans, lalu
lihat sampel dari tiap klaster -- supaya kelihatan apakah ini beberapa sub-kategori yang beda
(spam, scam generik, malvertising, dll), bukan satu kategori homogen.

In [ ]:
other_urls = train_df[train_df["label"] == "other"]["url"].astype(str)
if len(other_urls) > 50:
    other_tfidf = tfidf.transform(other_urls)
    k = 6
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    clusters = km.fit_predict(other_tfidf)
    print(f'Clustered {len(other_urls):,} "other" URLs into {k} groups. Sample from each:\n')
    for c in range(k):
        members = other_urls[clusters == c]
        print(f"--- Cluster {c} (n={len(members):,}, {len(members)/len(other_urls)*100:.1f}%) ---")
        for u in members.sample(min(5, len(members)), random_state=SEED):
            print(" ", u)
        print()
else:
    print('Too few "other" rows to cluster meaningfully.')

---
## §14 — Production Readiness Scorecard & Export

In [ ]:
TARGETS = {
    "F1-weighted": (">=", 0.95),
    "PR-AUC":      (">=", 0.95),
    "FPR":         ("<",  0.02),
    "Precision":   (">=", 0.96),
    "Recall":      (">=", 0.93),
}
row = res_df.loc[FINAL_MODEL_NAME]
print(f"Production-readiness scorecard -- model: {FINAL_MODEL_NAME}\n")
n_pass = 0
for metric, (op, target) in TARGETS.items():
    if metric not in row:
        continue
    val = row[metric]
    passed = (val >= target) if op == ">=" else (val < target)
    n_pass += int(passed)
    print(f"  {metric:14s} target {op} {target:<6}  actual={val:.4f}   {'PASS' if passed else 'FAIL'}")
print(f"\n  {n_pass}/{len(TARGETS)} aggregate targets met.")

### Cara baca scorecard + rekomendasi deploy

- FAIL di F1/PR-AUC/FPR agregat **diduga masih terjadi** (realistis, dari batas informasi
  lexical+char) -- bukan berarti pipeline gagal. Deploy sebagai **block + antrian review** (pakai
  `block_review` threshold di §8), bukan auto-block murni, sampai traffic production riil kasih
  data buat siklus retrain berikutnya.
- Kalau evasion rate di §11 tinggi (>15%) untuk perturbasi apa pun -- itu prasyarat yang harus
  ditangani (augmentasi data dengan variasi perturbasi itu) sebelum auto-block diaktifkan,
  terlepas dari angka F1/FPR di atas kelihatan bagus atau tidak.
- Klaster "other" di §13 memberi arah konkret kalau mau menambah sub-label granular untuk
  retrain berikutnya -- lihat sampel tiap klaster, itu petunjuk nyata, bukan tebakan leksikon.
- Fitur reputasi real-time (domain age, SSL) tetap di luar scope training ini -- lihat
  `apply_realtime_overrides()` di bawah untuk kontrak yang perlu diimplementasikan tim backend.

In [ ]:
with open(OUT_DIR / "label_encoder.json", "w") as f:
    json.dump({"classes": le.classes_.tolist(),
              "mapping": dict(zip(le.classes_, le.transform(le.classes_).tolist()))}, f)

with open(OUT_DIR / "feature_cols.json", "w") as f:
    json.dump({"feature_cols": LEX_COLS, "brands": BRANDS,
              "high_risk_tlds": sorted(HIGH_RISK_TLDS), "suspicious_kw": SUSPICIOUS_KW,
              "judol_kw": JUDOL_KW, "shorteners": sorted(SHORTENERS)}, f, indent=2)

with open(OUT_DIR / "lex_scaler.pkl", "wb") as f: pickle.dump(lex_scaler, f)
with open(OUT_DIR / "calibrators.pkl", "wb") as f: pickle.dump(cal_final, f)

cfg = {
    "final_model": FINAL_MODEL_NAME,
    "action_thresholds": ACTION_THRESHOLDS,
    "vocab_size": VOCAB_SIZE, "max_url_len": MAX_LEN,
    "char2idx": CHAR2IDX, "label_order": LABEL_ORDER,
    "char_encoder": WINNING_ENCODER_CLS.__name__,
    "tier0_cache_size": len(CACHE),
}
with open(OUT_DIR / "production_config.json", "w") as f:
    json.dump(cfg, f, indent=2)

res_df.to_csv(OUT_DIR / "results_full.csv")
print("Core artifacts saved.")

In [ ]:
import inspect

header_lines = [
    "# AFFERENT v5 -- feature_extractor.py",
    "# Auto-generated from the training notebook to guarantee training/serving parity.",
    "import re, math",
    "from collections import Counter",
    "from urllib.parse import urlparse",
    "import tldextract",
    "",
    f"BRANDS = {BRANDS!r}",
    f"HIGH_RISK_TLDS = {HIGH_RISK_TLDS!r}",
    f"SUSPICIOUS_KW = {SUSPICIOUS_KW!r}",
    f"JUDOL_KW = {JUDOL_KW!r}",
    f"SHORTENERS = {SHORTENERS!r}",
    "",
]

func_blocks = [inspect.getsource(fn) for fn in
              [levenshtein, brand_min_distance, brand_domain_mismatch,
               shannon_entropy, max_consecutive_consonants, extract_features]]

override_lines = [
    "",
    "def apply_realtime_overrides(base_prediction, live_features):",
    "    # STUB for the serving team -- NOT implemented or trained here.",
    "    # live_features is computed live by the backend (domain_age_days, ssl_valid,",
    "    # ssl_trust_score, redirect_count, asn_reputation, ...) -- point-in-time facts",
    "    # that cannot be reconstructed retroactively from a static training set.",
    "    #   domain_age = live_features.get('domain_age_days', 999)",
    "    #   lev_min = base_prediction.get('levenshtein_min', 99)",
    "    #   if domain_age < 3 and lev_min <= 2:",
    "    #       base_prediction['label'] = 'phishing'",
    "    #       base_prediction['confidence'] = max(base_prediction['confidence'], 0.70)",
    "    return base_prediction",
    "",
]

full_src = "\n".join(header_lines) + "\n\n" + "\n\n".join(func_blocks) + "\n" + "\n".join(override_lines)
with open(OUT_DIR / "feature_extractor.py", "w") as f:
    f.write(full_src)
print("Exported consolidated feature_extractor.py for serving-side reuse.")

print("\nAll artifacts saved to:", OUT_DIR)
for fp in sorted(OUT_DIR.iterdir()):
    if fp.is_file():
        print(f"  {fp.name:32s} {fp.stat().st_size/1024:8.1f} KB")

In [ ]:
print("\n" + "=" * 70)
print("AFFERENT v5 -- FINAL TRAINING SUMMARY")
print("=" * 70)
key = ["Logistic Regression", "Random Forest", "XGBoost (Optuna-tuned)",
      "TF-IDF + SGD (char n-grams)", "Char Encoder: BiLSTM (ablation)",
      "Char Encoder: Self-Attention (ablation)", "AfferentNet (unified multi-branch DL)",
      f"{FINAL_MODEL_NAME} [calibrated]"]
summary = res_df.loc[[m for m in key if m in res_df.index],
                     [c for c in ["Accuracy", "F1-weighted", "PR-AUC", "FPR"] if c in res_df.columns]]
print(summary.to_string())
print(f"\n>>> PRODUCTION MODEL: {FINAL_MODEL_NAME} <<<")
print(f"Action thresholds: {ACTION_THRESHOLDS}")
if p_ood is not None:
    print(f"\nOOD (Indonesian threats) -- {len(ood_df)} zero-shot URLs, "
          f"{(pred_ood != benign_idx).sum()} flagged malicious")
print("=" * 70)